In [ ]:
# Copyright 2026 International Business Machines

# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at

#     http://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 1.1 Introduction to EO Foundation Models

## Fine-Tuning with TerraMind

---

## 📚 Workshop Overview

This workshop introduces participants to **Earth Observation (EO) foundation models** and provides hands-on experience with fine-tuning and inference workflows.

In this notebook you will work with [TerraMind](https://huggingface.co/ibm-esa-geospatial) to:

- Understand EO foundation model concepts
- Perform a simple fine-tuning workflow 
- Run inference with pre-trained TerraMind models
- Compare single-modal (Sentinel-2) vs multi-modal (RGB+DEM) predictions

The session builds practical intuition for how EO foundation models are adapted to downstream tasks.  
  
Please see the Hugging Face page for more details of the saltmarsh models used in this session.

---

## 📋 Prerequisites

Please read the [instructions](https://github.com/IBM/ML4EO-workshop-2026) on the repository landing page.

---

## 0. Setup

### 0.1 Check Python Version

It's recommended that you run this notebook using Python 3.12.

In [ ]:
!python --version

### 0.2 Install and Import Dependencies

If running locally, ensure you have the necessary packages installed. This assumes you're in the project root directory.

Once installed, import dependencies.

In [ ]:
# Uncomment if you need to install dependencies
# TODO: ADD DEPENDENCIES
# !pip install uv
# !uv sync

In [ ]:
import os
import warnings
from pathlib import Path

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✓ Dependencies imported successfully!")

### 0.3 Download Sample Dataset

Before proceeding, you'll need to download the **sample dataset** for fine-tuning.

#### Download Sample Dataset from Google Drive

Download the sample salt marsh dataset and extract it to the `data/` directory:

```bash
# TODO: Add Google Drive download link and instructions
# Expected structure:
# data/saltmarsh_sample/
#   ├── train/
#   │   ├── images/
#   │   └── labels/
#   ├── val/
#   │   ├── images/
#   │   └── labels/
#   └── test/
#       ├── images/
#       └── labels/
```

### 0.4 Set Up Paths for Fine-Tuning

Define the paths we'll need for fine-tuning and testing.

In [ ]:
# Project root (adjust if running from different location)
project_root = Path("../..").resolve()
print(f"Project root: {project_root}")

# Local config file for fine-tuning (in repo)
config_file = project_root / "config/config-ibm-geospatial-saltmarsh-uk-s2-extent.yaml"

# Data paths (sample dataset)
data_root = project_root / "data/saltmarsh_sample"
train_images = data_root / "train/images"
train_labels = data_root / "train/labels"
val_images = data_root / "val/images"
val_labels = data_root / "val/labels"
test_images = data_root / "test/images"
test_labels = data_root / "test/labels"

# Output directory for fine-tuning
output_dir = project_root / "data/finetune/s2"
os.makedirs(output_dir, exist_ok=True)

print(f"\n✓ Paths configured for fine-tuning:")
print(f"  Config file: {config_file}")
print(f"  Data root: {data_root}")
print(f"  Output directory: {output_dir}")

---

## 1. Fine-Tuning

### 1.1 Introduction to Fine-Tuning

Fine-tuning is the process of adapting a pre-trained foundation model to a specific task or domain. This approach leverages the knowledge learned during pre-training while specializing the model for your use case, enabling a task-specific model to be created.


In this section, we'll fine-tune a TerraMind model on salt marsh extent detection using Sentinel-2 imagery.

### 1.2 Prepare Fine-Tuning Command

We'll use TerraTorch's command-line interface to fine-tune the model. The configuration file (local, in the repo) specifies all the training parameters.

For this demo, we'll:
- Train for **1 epoch** only
- Use a **batch size of 2** for faster execution
- Use the **sample dataset** (smaller subset for demonstration)

The model will be trained using the local config file which contains all the hyperparameters and model architecture details.

In [ ]:
# Construct the fine-tuning command
fine_tuning_command = f"""terratorch fit \
    --config {config_file} \
    --trainer.default_root_dir {output_dir} \
    --trainer.max_epochs 1 \
    --data.init_args.batch_size 2"""

print("Fine-tuning command:")
print(fine_tuning_command)
print("\nNote: Using max_epochs=1 and batch_size=2 for quick demo.")
print("For full training, increase epochs and adjust batch size based on available GPU memory.")

### 1.3 Run Fine-Tuning

Execute the fine-tuning command. This will train the model for 1 epoch and save checkpoints to the output directory.

**Note:** PyTorch Lightning automatically creates the directory structure:
- `lightning_logs/` - Lightning's log directory
- `version_0/` - First run (increments for subsequent runs)
- `checkpoints/` - Contains the `.ckpt` files

In [ ]:
# Run fine-tuning
!{fine_tuning_command}

### 1.4 Locate the New Checkpoint

After training, let's find the checkpoint that was just created.

In [ ]:
# Find the newly created checkpoint
new_checkpoint_dir = output_dir / "lightning_logs/version_0/checkpoints"
if new_checkpoint_dir.exists():
    new_checkpoints = list(new_checkpoint_dir.glob("*.ckpt"))
    if new_checkpoints:
        print(f"✓ Found {len(new_checkpoints)} checkpoint(s):")
        for ckpt in new_checkpoints:
            print(f"  - {ckpt.name}")
        # Use the first checkpoint for subsequent steps if needed
        new_checkpoint_path = new_checkpoints[0]
        print(f"\nUsing checkpoint: {new_checkpoint_path}")
    else:
        print("⚠ No checkpoints found in the directory.")
else:
    print(f"⚠ Checkpoint directory not found: {new_checkpoint_dir}")

---

## 2. Testing

### 2.1 Run Test on Fine-Tuned Model

Now we'll test the model we just fine-tuned on the test dataset.

This will evaluate the model's performance on held-out test data and report metrics.

In [ ]:
# Construct the test command using the newly fine-tuned checkpoint
test_command = f"""terratorch test \
    --config {config_file} \
    --ckpt_path {new_checkpoint_path} \
    --trainer.default_root_dir {output_dir}"""

print("Test command:")
print(test_command)

In [ ]:
# Run testing
!{test_command}

---

## 3. Inference with Pre-Trained Models

Now we'll download pre-trained models from HuggingFace and run inference to compare different approaches.

### 3.1 Download Pre-Trained Checkpoints from HuggingFace

We'll download two pre-trained models:
1. **S2 Extent Model** - Sentinel-2 based salt marsh extent detection
2. **RGB+DEM Extent Model** - Multi-modal (aerial RGB + DEM) salt marsh extent detection

In [ ]:
# Download S2 extent model checkpoint from HuggingFace
# TODO: Add actual HuggingFace model URL
s2_checkpoint_url = "https://huggingface.co/ibm-esa-geospatial/[MODEL_NAME]/resolve/main/checkpoint.ckpt"
s2_checkpoint_local = "./checkpoints/ibm-geospatial-saltmarsh-uk-s2-extent-10m_state_dict.ckpt"

# Download RGB+DEM extent model checkpoint from HuggingFace
# TODO: Add actual HuggingFace model URL
rgbdem_checkpoint_url = "https://huggingface.co/ibm-esa-geospatial/[MODEL_NAME]/resolve/main/checkpoint.ckpt"
rgbdem_checkpoint_local = "./checkpoints/ibm-geospatial-saltmarsh-uk-rgbdem-extent-2m_state_dict.ckpt"

# Create checkpoints directory
os.makedirs("./checkpoints", exist_ok=True)

# Uncomment to download (requires wget or curl)
# !wget -O {s2_checkpoint_local} {s2_checkpoint_url}
# !wget -O {rgbdem_checkpoint_local} {rgbdem_checkpoint_url}

print("✓ Download instructions ready!")
print(f"S2 checkpoint will be saved to: {s2_checkpoint_local}")
print(f"RGB+DEM checkpoint will be saved to: {rgbdem_checkpoint_local}")

### 3.2 Set Up Inference Paths

Define paths for inference outputs and data.

In [ ]:
# Checkpoint paths
s2_checkpoint_path = Path(s2_checkpoint_local)
rgbdem_checkpoint_path = Path(rgbdem_checkpoint_local)

# Inference data paths
inference_images_s2 = project_root / "data/inference/images/s2"
inference_images_aerial = project_root / "data/inference/images/aerial"

# Output directories for inference
s2_inference_output = project_root / "data/inference/pred/s2"
rgbdem_inference_output = project_root / "data/inference/pred/rgb_dem_extent"

# Create output directories
os.makedirs(s2_inference_output, exist_ok=True)
os.makedirs(rgbdem_inference_output, exist_ok=True)

print(f"\n✓ Inference paths configured:")
print(f"  S2 checkpoint: {s2_checkpoint_path}")
print(f"  RGB+DEM checkpoint: {rgbdem_checkpoint_path}")
print(f"  S2 inference output: {s2_inference_output}")
print(f"  RGB+DEM inference output: {rgbdem_inference_output}")

### 3.3 Run Inference with Sentinel-2 Model

Run prediction on inference images using the pre-trained S2 checkpoint.

For inference, we'll use the **config from HuggingFace** rather than the local config used for fine-tuning.

In [ ]:
# TODO: Add actual HuggingFace config path
s2_config_hf = "hf://ibm-esa-geospatial/[MODEL_NAME]/config.yaml"

# Construct the prediction command
s2_predict_command = f"""terratorch predict \
    --config {s2_config_hf} \
    --ckpt_path {s2_checkpoint_path} \
    --trainer.default_root_dir {output_dir} \
    --predict_output_dir {s2_inference_output} \
    --data.init_args.predict_data_root.S2L2A {inference_images_s2}"""

print("S2 Prediction command:")
print(s2_predict_command)

In [ ]:
# Run S2 prediction
!{s2_predict_command}

### 3.4 Visualize S2 Results

Let's visualize the predictions from the S2 model.

**TODO**: Add visualization functions to display:
- Input Sentinel-2 imagery
- Model predictions
- Comparison overlays

In [ ]:
# TODO: Add visualization code
# Example structure:
# - Load prediction results from s2_inference_output
# - Load corresponding inference images
# - Create side-by-side plots
# - Calculate and display metrics if ground truth available

print("Visualization functions to be added here.")

---

## 4. Multi-Modal Inference

### 4.1 Run Inference with RGB+DEM Model

Now let's run inference using a multi-modal model that combines aerial RGB imagery with Digital Elevation Model (DEM) data.

This model uses:
- **RGB**: High-resolution aerial imagery (2m resolution)
- **DEM**: Digital Elevation Model for terrain information

We'll use the **config from HuggingFace** for this model as well.

In [ ]:
# TODO: Add actual HuggingFace config path
rgbdem_config_hf = "hf://ibm-esa-geospatial/[MODEL_NAME]/config.yaml"

# Construct the prediction command for RGB+DEM model
rgbdem_predict_command = f"""terratorch predict \
    --config {rgbdem_config_hf} \
    --ckpt_path {rgbdem_checkpoint_path} \
    --trainer.default_root_dir {output_dir} \
    --predict_output_dir {rgbdem_inference_output} \
    --data.init_args.predict_data_root.RGB {inference_images_aerial} \
    --data.init_args.predict_data_root.DEM {inference_images_aerial}"""

print("RGB+DEM Prediction command:")
print(rgbdem_predict_command)

In [ ]:
# Run RGB+DEM prediction
!{rgbdem_predict_command}

### 4.2 Compare S2 vs RGB+DEM Models

Let's compare the predictions from both models to understand the benefits of multi-modal approaches.

**TODO**: Add comparison visualization showing:
- Side-by-side predictions from S2 and RGB+DEM models
- Difference maps highlighting where models disagree
- Quantitative metrics comparison
- Discussion of when each model performs better

In [ ]:
# TODO: Add comparison visualization code
# Example structure:
# - Load predictions from both models
# - Create comparison plots
# - Calculate metrics for both models
# - Highlight areas where multi-modal approach excels

print("Model comparison visualization to be added here.")

### Key Insights: Multi-Modal vs Single-Modal

Multi-modal models (RGB+DEM) often provide advantages:

- **Higher Resolution**: Aerial imagery at 2m vs Sentinel-2 at 10m
- **Terrain Context**: DEM provides elevation information crucial for salt marsh identification
- **Complementary Information**: RGB captures visual features while DEM captures topographic features
- **Improved Accuracy**: Combining modalities typically improves detection performance

However, single-modal S2 models have benefits too:

- **Global Coverage**: Sentinel-2 provides consistent global coverage
- **Temporal Resolution**: Regular revisit times for monitoring changes
- **Free and Open**: Publicly available data
- **Spectral Bands**: Multiple spectral bands beyond RGB

---

## 📖 Next Steps

Continue to the next notebook:

- **1.2 Multimodal Inference with TerraMind** - Learn more about running inference with different data modalities

---

## 🔗 Additional Resources

- [TerraMind on HuggingFace](https://huggingface.co/ibm-esa-geospatial)
- [TerraTorch Documentation](https://github.com/IBM/terratorch)
- [TerraKit Documentation](https://torchgeo.org/terrakit/)
- [Prithvi Model Hub](https://huggingface.co/ibm-nasa-geospatial)

---

<!--
Copyright (c) 2026 IBM Corp

This software is released under the MIT License.
https://opensource.org/licenses/MIT
-->